# Day 2.5 — Citations and Abstention
A free-text answer cannot be checked and a free-text refusal cannot be acted on. We now demand a
structured answer whose citations name chunk ids *we* supplied, then verify them ourselves: a
citation is a claim about our own evidence.

### Step 1 — The contract, and the field we refuse to ask for

`ModelAnswer` is exactly what the model may return. Notice what is missing: `grounded`. A model
declaring its own answer trustworthy adds no information, so the application computes that field
afterwards on `GroundedAnswer`.

In [ ]:
class Citation(BaseModel):
    source: str
    section: str
    chunk_id: str

class ModelAnswer(BaseModel):
    """Exactly the fields we ask the model for - no self-assessment."""
    answer: str
    citations: list[Citation] = Field(default_factory=list)
    abstained: bool

class GroundedAnswer(ModelAnswer):
    """A model answer AFTER the application has checked it."""
    grounded: bool = False                       # set by validate_citations, never by the model
    dropped_citations: list[Citation] = Field(default_factory=list)

ANSWER_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "grounded_answer", "strict": True, "schema": make_strict(ModelAnswer.model_json_schema())}}

schema = ANSWER_FORMAT["json_schema"]["schema"]
print("top level closed to extra keys :", schema["additionalProperties"] is False)
print("every property required        :", sorted(schema["properties"]) == sorted(schema["required"]))
print("nested Citation closed         :", schema["$defs"]["Citation"]["additionalProperties"] is False)
print("model asked to certify itself? :", "grounded" in schema["properties"])
print("fields the model may return    :", sorted(schema["properties"]))

### Step 2 — Ask with the schema, then validate the citations yourself

Same retrieval, same prompt, one addition: `response_format`. Then keep only citations whose
`chunk_id` we retrieved *and* whose source and section match that chunk. Whatever survives sets
`grounded`; an abstention is well formed only when it cites nothing at all.

In [ ]:
def validate_citations(answer, retrieved):
    """Drop citations we cannot back with our own retrieval log, then set `grounded`."""
    supplied = {item.chunk.chunk_id: item.chunk for item in retrieved}
    kept, dropped = [], []
    for citation in answer.citations:
        chunk = supplied.get(citation.chunk_id)
        if chunk is not None and chunk.source == citation.source and chunk.section == citation.section:
            kept.append(citation)                    # we supplied it, and the labels match
        else:
            dropped.append(citation)                 # invented, or copied from another answer
    answer.citations, answer.dropped_citations = kept, dropped
    answer.grounded = (not kept and not dropped) if answer.abstained else (bool(kept) and not dropped)
    return answer

reply = chat([{"role": "user", "content": build_prompt(QUESTION, build_context(retrieved, CONTEXT_BUDGET))}],
             response_format=ANSWER_FORMAT)
answer = validate_citations(GroundedAnswer(**ModelAnswer.model_validate_json(reply["content"]).model_dump()), retrieved)

print("supplied to the model :", [item.chunk.chunk_id for item in retrieved])
print("abstained             :", answer.abstained)
print("kept citations        :", [c.chunk_id for c in answer.citations])
print("dropped citations     :", [c.chunk_id for c in answer.dropped_citations])
print("grounded              :", answer.grounded)
print("answer                :", answer.answer[:150], "...")
print("\ngrounded=True means: not an abstention, at least one citation survived, nothing was dropped.")
print("It is computed from our own retrieval log, so the model cannot fake it.")

### Step 3 — Break it twice: an invented citation, and a missing answer

Models do return citations that were never supplied — copied from an earlier answer, or simply
plausible. And the unanswerable question now has a defined outcome: `abstained=True` with zero
citations, a *successful* result rather than an error.

In [ ]:
tampered = GroundedAnswer(
    answer="Fault records are retained for one year.",
    citations=[Citation(source="battery_safety.md", section="Data retention", chunk_id="battery_safety:data-retention"),
               Citation(source="battery_safety.md", section="Appendix C", chunk_id="battery_safety:appendix-c")],  # never existed
    abstained=False)
checked = validate_citations(tampered, retrieved)
print("INVENTED CITATION")
print("  kept    :", [c.chunk_id for c in checked.citations])
print("  dropped :", [c.chunk_id for c in checked.dropped_citations])
print("  grounded:", checked.grounded, "-> a caller can refuse to display this answer")

missing = BEST_INDEX.search(UNANSWERABLE, top_k=3)
refusal_reply = chat([{"role": "user", "content": build_prompt(UNANSWERABLE, build_context(missing, CONTEXT_BUDGET))}],
                     response_format=ANSWER_FORMAT)
refusal = validate_citations(GroundedAnswer(**ModelAnswer.model_validate_json(refusal_reply["content"]).model_dump()), missing)
print("\nUNANSWERABLE QUESTION")
print("  retrieved anyway:", [item.chunk.chunk_id for item in missing])
print("  abstained       :", refusal.abstained, "| citations:", refusal.citations)
print("  grounded        :", refusal.grounded, "(an abstention is grounded when it cites nothing)")
print("  answer          :", refusal.answer)

### Step 4 — The whole pipeline behind one call

Retrieve, assemble, generate, validate, and record every intermediate result in a `KnowledgeState`
the application owns. This object *is* the Engineering Knowledge Assistant; the rest of the day
measures and improves it.

In [ ]:
from typing import Literal, Optional

class KnowledgeState(BaseModel):
    """Everything the application knows about one run - the model never sees this."""
    question: str
    retrieved: list[Retrieved] = Field(default_factory=list)
    answer: Optional[GroundedAnswer] = None
    status: Literal["created", "retrieved", "completed", "failed"] = "created"
    error: Optional[str] = None

class KnowledgeAssistant:
    def __init__(self, index, top_k=3, budget=CONTEXT_BUDGET):
        self.index, self.top_k, self.budget = index, top_k, budget

    def answer(self, question):
        state = KnowledgeState(question=question)
        try:
            state.retrieved = self.index.search(question, top_k=self.top_k)
            state.status = "retrieved"
            prompt = build_prompt(question, build_context(state.retrieved, self.budget))
            reply = chat([{"role": "user", "content": prompt}], response_format=ANSWER_FORMAT)
            model_answer = ModelAnswer.model_validate_json(reply["content"])
            state.answer = validate_citations(GroundedAnswer(**model_answer.model_dump()), state.retrieved)
            state.status = "completed"
        except Exception as error:                    # one bad reply must not stop a batch of ten
            state.status, state.error = "failed", f"{type(error).__name__}: {error}"
        return state

assistant = KnowledgeAssistant(BEST_INDEX, top_k=3)
state = assistant.answer("Does requesting island mode immediately open the grid breaker?")
print("status   :", state.status, "| retrieved:", [item.chunk.chunk_id for item in state.retrieved])
print("grounded :", state.answer.grounded, "| citations:", [c.chunk_id for c in state.answer.citations])
print("answer   :", state.answer.answer[:200], "...")

### Try it yourself

Invent two questions the corpus cannot answer — not about price. Predict whether the assistant
abstains, then check; the second exposes the limits of the mock.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
for my_question in ["What is the wifi password for the campus network?",
                    "Who manufactured the battery cells?"]:
    result = assistant.answer(my_question)
    print(my_question)
    print(f"   abstained={result.answer.abstained}  citations={[c.chunk_id for c in result.answer.citations]}")
print()
print("The first abstains: no retrieved passage contains 'wifi' or 'password', and there is no list")
print("of forbidden topics anywhere in our code - the decision comes from the evidence.")
print("The second can fool the mock: the battery text contains the word 'cell', so the lexical rule")
print("believes it has evidence about who made them. A real model reads the passage, sees no")
print("manufacturer and abstains - one good reason to run this cell again with a key.")

### Checkpoint

**1. The model returned `"grounded": true`. Why do we not even offer it that field?**

<details><summary>Show answer</summary>

It comes from the same process that produced the answer, so it adds no independent information: a model that invents a citation will also claim to be grounded. Our flag comes from the retrieval log we kept.

</details>

**2. An abstention arrives with two citations attached. Is that acceptable?**

<details><summary>Show answer</summary>

No, and `validate_citations` marks it `grounded=False`. Abstention means the evidence does not support an answer; attaching sources contradicts that. An answer cites; an abstention does not.

</details>

### Recap

- **Limitation seen:** free-text answers and refusals cannot be checked, and citations can name chunks we never retrieved.
- **Layer added:** a strict answer schema, citation validation that sets `grounded`, and the assembled assistant.
- **Evidence:** the invented `battery_safety:appendix-c` citation was dropped and flagged; the unanswerable question abstained with zero citations.